## The configuration problem

You ended notebook 03 with a Deployment running an image that bakes in a default config. Now you want to run the *same* image in staging with `LOG_LEVEL=debug` and in prod with `LOG_LEVEL=warn`, point a *web* pod at a swappable backend URL, and mount one TLS cert into three services.

Options that **don't** work in production:

- **Bake it into the image.** You'd build `web:staging` and `web:prod` — different images, different test coverage. Promotion becomes "build a new image" instead of "redeploy the same one with new config" — so the image you tested isn't the one you ship.
- **Pass it on the command line.** Where you came from in Docker. But pods are launched by the kubelet from a `template` in `etcd`, not by you at a prompt. One variable inline is fine; twenty makes every Deployment a wall of env entries.
- **Stash it in a sidecar.** Sometimes right, but heavy for "put a value in an env var."

What you want is a **first-class object** that holds config, lives in the cluster like any other object, and is *referenced* from pod templates. Two exist:

- **`ConfigMap`** — non-sensitive data.
- **`Secret`** — sensitive data.

Near-identical schemas and consumption; the differences are about *handling* (base64 storage, encryption at rest, RBAC), not shape. Master ConfigMap and Secret is the same plus a security posture. On our map, both are chips in the **Config** box — the layer that keeps configuration *out* of the image and *in* the cluster.